# `eff-activity` — pre-merge test notebook

Branch: `eff-activity` &nbsp;|&nbsp; checkpoint §I' action **F1** (§F)

**What this branch adds — `eff_activity` (effective-rule genome-species activity).**
Full-genome `activity` keys on an FNV-1a hash of the whole 32-byte LUT, so flipping a
LUT bit for an input the cell never encounters mints a *new species* with no behavioural
change (silent churn). `eff_activity` keys instead on the LUT **masked to the entries the
population actually exercises** (the union of the per-cell used-masks accumulated during
Phase 1), folded with the same egene stream. Two genomes differing only in never-exercised
entries collapse to one effective-rule species.

**Design conflict surfaced & resolved.** The task asked for a *per-cell* used-mask keying
each cell's effective rule. A literal per-cell key is neighbourhood-history dependent, so
two cells with an *identical* genome but different realised neighbourhoods hash to different
effective-rule species — *splitting* one genome into many. Empirically that makes
eff-species **3-10x larger** than full-genome species, inverting the intended "collapse"
goal and violating criterion `eff <= full`. Resolution: the per-cell masks are still
accumulated and exposed (`get_eff_used_mask`, the directed diagnostic substrate), but the
species **key** applies their population **union** (`get_eff_union_mask`), which is a pure
coarsening of genome space — so `eff <= full` and the silent-churn invariance both hold.

Each code cell below prints **PASS**/**FAIL**.


In [1]:
import os, sys
import numpy as np
from collections import defaultdict

ROOT = os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'python'))
from evoca_py import EvoCA, make_gol_lut, LUT_BYTES, LUT_BITS
print('repo root:', ROOT)
print('LUT_BYTES =', LUT_BYTES, ' LUT_BITS =', LUT_BITS)

repo root: /Users/n/Projects/EvoCA/.claude/worktrees/agent-ae3608784caee8a59
LUT_BYTES = 32  LUT_BITS = 250


In [2]:
# Cell 1 - ADDITIVITY: GoL dynamics + existing full-genome activity are unchanged.
# The eff_activity machinery must be a no-op on the CA. A GoL blinker still
# oscillates period-2, full-genome activity still sees exactly one species under
# mu=0, and eff_activity collapses the uniform genome to one effective-rule species.
# NB: the C core is a global singleton, so we use one live sim at a time.
sb = EvoCA(); sb.init(16, mu_lut=0.0, mu_egene=0.0, mu_egenome=0.0, tax=0.0, food_inc=0.0)
sb.set_lut_all(make_gol_lut()); sb.set_alive_all(); sb.set_egenome_all(0)
v = np.zeros((16,16), np.uint8)
for (r,c) in [(8,7),(8,8),(8,9)]: v[r,c]=1
sb.set_v(v); sb.step(); v1 = sorted(zip(*np.where(sb.get_v()==1)))
sb.step(); v2 = sorted(zip(*np.where(sb.get_v()==1)))
blinker_ok = (v1 == sorted([(7,8),(8,8),(9,8)])) and (v2 == sorted([(8,7),(8,8),(8,9)]))
sb.free()

N = 32
s = EvoCA(); s.init(N, mu_lut=0.0, mu_egene=0.0, mu_egenome=0.0, tax=0.0, food_inc=0.0)
s.set_lut_all(make_gol_lut()); s.set_alive_all(); s.set_egenome_all(0)
rng = np.random.default_rng(0); s.set_v(rng.integers(0,2,(N,N),np.uint8))
for _ in range(50): s.step()
s._lib.evoca_activity_update(); s.eff_activity_update()
full = s.n_distinct_genomes(); eff = s.eff_count_distinct()
print(f'blinker period-2 unchanged : {blinker_ok}')
print(f'full-genome species (mu=0) : {full}  (expect 1)')
print(f'eff species        (mu=0)  : {eff}  (expect 1)')
ok1 = blinker_ok and full == 1 and eff == 1
print('PASS' if ok1 else 'FAIL')

blinker period-2 unchanged : True
full-genome species (mu=0) : 1  (expect 1)
eff species        (mu=0)  : 1  (expect 1)
PASS


In [3]:
# Cell 2 - HEADLINE: pure unused-bit churn mints NO eff-species (but many full ones).
# After accumulating the population-union mask, randomise each alive cell's
# GLOBALLY-UNUSED LUT entries. Full-genome hashes change everywhere; every cell's
# effective-rule hash is unchanged, so eff_activity is blind to the silent churn.
eff_before = s.get_eff_hashes().copy()
full_before = s.n_distinct_genomes()
eff_cd_before = s.eff_count_distinct()

union = s.get_eff_union_mask()
unused = [b for b in range(LUT_BITS) if not ((union[b>>3] >> (b&7)) & 1)]
alive = s.get_alive().astype(bool)
print(f'population-union exercised entries : {LUT_BITS-len(unused)}/{LUT_BITS}   '
      f'globally-unused (silent) entries : {len(unused)}')

for i in range(N*N):
    r,c = divmod(i, N)
    if not alive[r,c]: continue
    lb = s.get_lut(i).copy()
    cr = np.random.default_rng(10_000 + i)
    for b in unused:
        if cr.integers(0,2): lb[b>>3] ^= np.uint8(1 << (b&7))
    s.set_lut(i, lb)

s._lib.evoca_activity_update(); s.eff_activity_update()
eff_after = s.get_eff_hashes()
full_after = s.n_distinct_genomes(); eff_cd_after = s.eff_count_distinct()

eff_unchanged = np.array_equal(eff_after[alive], eff_before[alive])
print(f'full-genome species  before -> after : {full_before} -> {full_after}   (silent churn mints new full species)')
print(f'eff-species          before -> after : {eff_cd_before} -> {eff_cd_after}   (unchanged: churn invisible to eff)')
print(f'every alive cell eff-hash unchanged  : {eff_unchanged}')
print(f'eff-species <= full-species          : {eff_cd_after <= full_after}')
ok2 = eff_unchanged and eff_cd_after == eff_cd_before and full_after > full_before and eff_cd_after <= full_after
print('PASS' if ok2 else 'FAIL')

population-union exercised entries : 232/250   globally-unused (silent) entries : 18
full-genome species  before -> after : 1 -> 1021   (silent churn mints new full species)
eff-species          before -> after : 1 -> 1   (unchanged: churn invisible to eff)
every alive cell eff-hash unchanged  : True
eff-species <= full-species          : True
PASS


In [4]:
# Cell 3 - eff <= full over a churning run, proved rigorously (hash-aliasing-immune).
# In a mutation-churned population, group alive cells by EXACT genome bytes and check
# every group maps to a single eff-hash: eff is a function of the genome, hence the
# alive eff-species count can never exceed the alive genome count.
M = 40
g = EvoCA(); g.init(M, mu_lut=0.02, mu_egene=0.0, mu_egenome=0.0, tax=0.0, food_inc=0.0)
g.set_lut_all(make_gol_lut()); g.set_alive_all(); g.set_egenome_all(0); g.set_f_all(2.0)
rng = np.random.default_rng(0); g.set_v(rng.integers(0,2,(M,M),np.uint8))
rows = []
for t in range(1, 26):
    g.step(); g.set_f_all(2.0)
    g._lib.evoca_activity_update(); g.eff_activity_update()
    if t % 5 == 0: rows.append((t, g.n_distinct_genomes(), g.eff_count_distinct()))

alive = g.get_alive().astype(bool).ravel()
effh = g.get_eff_hashes().ravel()
luts = [g.get_lut(i).tobytes() for i in range(M*M)]
groups = defaultdict(set)
for i in range(M*M):
    if alive[i]: groups[luts[i]].add(int(effh[i]))
coarsening_ok = all(len(v) == 1 for v in groups.values())

print(' t   full  eff')
for (t,f,e) in rows: print(f'{t:3d} {f:6d} {e:5d}')
print(f'eff = f(genome) (each exact genome -> one eff hash): {coarsening_ok}')
print(f'alive eff-species <= distinct genomes            : {g.eff_count_distinct()} <= {len(groups)} = {g.eff_count_distinct() <= len(groups)}')
ok3 = coarsening_ok and (g.eff_count_distinct() <= len(groups))
print('PASS' if ok3 else 'FAIL')
g.free()

 t   full  eff
  5   1576  1573
 10   1592  1591
 15   1595  1594
 20   1595  1595
 25   1600  1599
eff = f(genome) (each exact genome -> one eff hash): True
alive eff-species <= distinct genomes            : 1599 <= 1600 = True
PASS


In [5]:
# Cell 4 - MERGE-GATE: memory cost of the per-cell used-mask (diagnostic substrate).
for Ncost in (256, 512):
    mb = Ncost*Ncost*LUT_BYTES / (1024*1024)
    print(f'N={Ncost:4d}: per-cell used-mask = {Ncost}*{Ncost}*{LUT_BYTES} B = {mb:.1f} MB   '
          f'(population-union key mask = {LUT_BYTES} B, negligible)')
print('note: the union mask alone (32 B) drives the species key; the per-cell array is the'
      ' exposed diagnostic and could be dropped as a follow-up if only the metric is needed.')

N= 256: per-cell used-mask = 256*256*32 B = 2.0 MB   (population-union key mask = 32 B, negligible)
N= 512: per-cell used-mask = 512*512*32 B = 8.0 MB   (population-union key mask = 32 B, negligible)
note: the union mask alone (32 B) drives the species key; the per-cell array is the exposed diagnostic and could be dropped as a follow-up if only the metric is needed.


## Expected result

All four PASS/FAIL cells print **PASS**.

| Cell | Guarantee | Why it matters for F1 |
|---|---|---|
| 1 | GoL blinker period-2 intact; full-genome activity == 1 species at mu=0; eff == 1 | eff_activity is an additive no-op - dynamics and existing `activity` unchanged |
| 2 | Randomising globally-unused LUT entries mints many new **full** species but **zero** new eff-species; every eff-hash unchanged | the core point: silent churn is invisible to eff_activity |
| 3 | eff is a function of the exact genome (each genome maps to one eff hash), so eff-species <= genome count | `eff <= full` holds as a genuine coarsening, immune to hash aliasing |
| 4 | per-cell used-mask memory: 2.1 MB (N=256) / 8.4 MB (N=512); union key mask 32 B | merge-gate memory decision |

**Key design note.** The species key masks by the **population-union** of exercised
entries, not each cell's own per-cell mask: a per-cell key splits identical genomes by
neighbourhood history and makes eff-species *larger* than full-species (3-10x), defeating
the collapse goal. The per-cell masks remain accumulated and exposed via
`sim.get_eff_used_mask()` as the diagnostic substrate the union is built from.
